1. Setup Project structure
2. S&P500 Ticker CSV File Construction
3. Data_loader
4. Feature Engineering
5. Data set construction
6. Models
7. Evaluate
8. Train
9. Result

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_DIR = "/content/drive/MyDrive/ml_asset_pricing"

dirs = [
    f"{BASE_DIR}/data/raw",
    f"{BASE_DIR}/data/processed",
    f"{BASE_DIR}/src/models",
    f"{BASE_DIR}/notebooks",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("folder created in Drive:", BASE_DIR)

folder created in Drive: /content/drive/MyDrive/ml_asset_pricing


In [4]:
open(f"{BASE_DIR}/src/__init__.py", "a").close()
open(f"{BASE_DIR}/src/models/__init__.py", "a").close()

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import requests
from io import StringIO

def save_sp500_csv_once(
    path="/content/drive/MyDrive/ml_asset_pricing/data/sp500_tickers_2026.csv"
):
    #  이미 있으면 절대 안 만듦
    if os.path.exists(path):
        print("CSV already exists → skip download")
        return

    print("Creating CSV...")

    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}

    res = requests.get(url, headers=headers)
    tables = pd.read_html(StringIO(res.text))

    df = tables[0]
    tickers = df["Symbol"].tolist()
    tickers = [t.replace(".", "-") for t in tickers]

    # 폴더 없으면 생성
    os.makedirs(os.path.dirname(path), exist_ok=True)

    pd.DataFrame({"ticker": tickers}).to_csv(path, index=False)

    print(f"Saved {len(tickers)} tickers → {path}")


save_sp500_csv_once()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSV already exists → skip download


In [ ]:
%%writefile /content/drive/MyDrive/ml_asset_pricing/src/data_loader.py
import yfinance as yf
import pandas as pd
import requests


def get_sp500_tickers(path="/content/drive/MyDrive/ml_asset_pricing/data/sp500_tickers_2026.csv"):

    df = pd.read_csv(path)

    # 컬럼 이름이 없으면 첫 컬럼 사용
    tickers = df.iloc[:, 0].tolist()

    tickers = [t.replace(".", "-") for t in tickers]

    return tickers

def download_data(tickers, start="2010-01-01", end="2024-12-31"):

    raw = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=True, # Close가 Adj Close(조정 종가): 배당 + 액면분할까지 반영된 가격 => 수익률 계산 시 사용
        progress=False
    )

    # 조용한 실패를 막는 장치
    if raw.empty:
        raise ValueError("No data returned from yfinance.")

    # 여러 티커면 MultiIndex, 한 티커면 일반 DataFrame
    if isinstance(raw.columns, pd.MultiIndex):
        close = raw["Close"].copy()
    else:
        close = raw[["Close"]].copy()
        if isinstance(tickers, str):
            close.columns = [tickers]

    panel = close.stack().reset_index()
    panel.columns = ["date", "ticker", "price"]

    panel["date"] = pd.to_datetime(panel["date"])
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)

    panel["return"] = panel.groupby("ticker")["price"].pct_change()

    panel = panel.replace([float("inf"), float("-inf")], pd.NA)
    panel = panel.dropna(subset=["price", "return"]).reset_index(drop=True)

    # 티커별 데이터 개수 체크 & 데이터 정상 확인
   # counts = panel.groupby("ticker").size()
   # print(counts.describe())
   # print(panel["return"].describe())
   # print("티커별 데이터 정상인지 확인")

    return panel


def load_data():

    tickers = get_sp500_tickers()

    df = download_data(tickers)

    return df


def sample_universe(df, n=100, method="random"):

    def _sample(group):
        group = group.dropna()

        if len(group) < n:
            return group  # 부족하면 그냥 사용

        if method == "random":
            return group.sample(n=n, random_state=42)

        elif method == "topcap":
            return group.sort_values("market_cap", ascending=False).head(n)

        else:
            raise ValueError("method must be random or topcap")

    df = df.groupby("date", group_keys=False).apply(_sample)
    return df

def load_data_fixed(tickers, start="2010-01-01", end="2024-12-31"):
    raw = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )

    if raw.empty:
        raise ValueError("No data returned from yfinance.")

    if isinstance(raw.columns, pd.MultiIndex):
        close = raw["Close"].copy()
    else:
        close = raw[["Close"]].copy()
        if isinstance(tickers, str):
            close.columns = [tickers]

    panel = close.stack().reset_index()
    panel.columns = ["date", "ticker", "price"]

    panel["date"] = pd.to_datetime(panel["date"])
    panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)

    panel["return"] = panel.groupby("ticker")["price"].pct_change()

    panel = panel.replace([float("inf"), float("-inf")], pd.NA)
    panel = panel.dropna(subset=["price", "return"]).reset_index(drop=True)

    return panel

Overwriting /content/drive/MyDrive/ml_asset_pricing/src/data_loader.py


In [ ]:
%%writefile /content/drive/MyDrive/ml_asset_pricing/src/feature_engineering.py

import pandas as pd
import numpy as np

def add_features(df,mode="baseline"):
    df = df.sort_values(["ticker", "date"]).copy()

    # =========================
    # 1. PRICE TREND
    # =========================

    # Momentum
    df["mom_1m"] = df.groupby("ticker")["price"].pct_change(21)
    df["mom_3m"] = df.groupby("ticker")["price"].pct_change(63)
    df["mom_6m"] = df.groupby("ticker")["price"].pct_change(126)
    df["mom_12m"] = df.groupby("ticker")["price"].pct_change(252)

    # Momentum spread
    df["mom_3m_12m"] = df["mom_3m"] - df["mom_12m"]

    # Short-term reversal
    df["rev_1w"] = df.groupby("ticker")["return"].shift(5)
    df["rev_1m"] = df.groupby("ticker")["return"].shift(21)

    #log
    df["log_price"] = np.log(df["price"])

    # =========================
    # 2. VOLATILITY / RISK
    # =========================

    df["vol_1m"] = df.groupby("ticker")["return"].rolling(21).std().reset_index(level=0, drop=True)
    df["vol_3m"] = df.groupby("ticker")["return"].rolling(63).std().reset_index(level=0, drop=True)
    df["vol_6m"] = df.groupby("ticker")["return"].rolling(126).std().reset_index(level=0, drop=True)

    # downside volatility
    df["downside_vol"] = df.groupby("ticker")["return"].apply(
        lambda x: x.rolling(21).apply(lambda r: r[r < 0].std() if len(r[r < 0]) > 0 else 0)
    ).reset_index(level=0, drop=True)

    # =========================
    # 3. LIQUIDITY (proxy)
    # =========================

    # price 기반 proxy (실제는 volume 필요)
    df["price_level"] = df["price"]

    # turnover proxy (근사)
    df["ret_abs"] = df["return"].abs()
    df["illiq_proxy"] = df["ret_abs"] / df["price"]

    #log
    df["log_illiq"] = np.log(df["illiq_proxy"] + 1e-8)

    # =========================
    # 4. TREND / MEAN REVERSION MIX
    # =========================

    # moving average
    df["ma_20"] = df.groupby("ticker")["price"].transform(lambda x: x.rolling(20).mean())
    df["ma_60"] = df.groupby("ticker")["price"].transform(lambda x: x.rolling(60).mean())

    df["ma_ratio_20"] = df["price"] / df["ma_20"]
    df["ma_ratio_60"] = df["price"] / df["ma_60"]

    # =========================
    # 5. RETURN LAGS
    # =========================

    df["ret_lag_1"] = df.groupby("ticker")["return"].shift(1)
    df["ret_lag_2"] = df.groupby("ticker")["return"].shift(2)
    df["ret_lag_5"] = df.groupby("ticker")["return"].shift(5)
    df["ret_lag_10"] = df.groupby("ticker")["return"].shift(10)
    df["ret_lag_21"] = df.groupby("ticker")["return"].shift(21)


    if mode == "improved":
    # =========================
    # 6. interaction feature (데이터 수가 적음 -> 직접 만듦(다른 카테고리끼리))
    # =========================

    # Momentum × Volatility
      df["mom_vol_1"] = df["mom_3m"] * df["vol_1m"]
      df["mom_vol_2"] = df["mom_6m"] * df["vol_3m"]

    # Momentum × Reversal
      df["mom_rev"] = df["mom_3m"] * df["rev_1m"]

    # Volatility × Downside
      df["vol_down"] = df["vol_1m"] * df["downside_vol"]

    # Reversal × Volatility
      df["rev_vol"] = df["rev_1m"] * df["vol_1m"]

    # Liquidity proxy × Momentum
      df["illiq_mom"] = df["illiq_proxy"] * df["mom_3m"]

      df["mom_logvol"] = df["mom_3m"] * np.log(df["vol_1m"] + 1e-8)

    #특정 조건에서만 작동하는 signal(regime-dependent signal)
      df["high_vol_flag"] = (df["vol_1m"] > df["vol_1m"].median()).astype(int)
      df["mom_high_vol"] = df["mom_3m"] * df["high_vol_flag"]

    return df

Overwriting /content/drive/MyDrive/ml_asset_pricing/src/feature_engineering.py


In [ ]:
%%writefile /content/drive/MyDrive/ml_asset_pricing/src/dataset.py
import pandas as pd

# target 데이터
def create_target(df):
    df = df.sort_values(["ticker", "date"]).copy()
    df["target"] = df.groupby("ticker")["return"].shift(-1)
    return df

# 같은 날짜 안에서 종목끼리 비교, 각 feature를 순위 기반으로 정규화
def normalize_features(df, feature_cols):
    df = df.copy()
    # cross-sectional rank normalization by date
    df[feature_cols] = df.groupby("date")[feature_cols].rank(pct=True)
    return df

# train(학습용)/val(튜닝용)/test(평가) 데이터 셋 나누기
def split_data(df, train_end="2015-12-31", val_end="2017-12-31"):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    train_end = pd.Timestamp(train_end)
    val_end = pd.Timestamp(val_end)

    train = df[df["date"] <= train_end].copy()
    val = df[(df["date"] > train_end) & (df["date"] <= val_end)].copy()
    test = df[df["date"] > val_end].copy()

    return train, val, test

def get_feature_cols(df):
    exclude = {"date", "ticker", "price", "return", "target"}
    return [c for c in df.columns if c not in exclude]

Overwriting /content/drive/MyDrive/ml_asset_pricing/src/dataset.py


In [ ]:
%%writefile /content/drive/MyDrive/ml_asset_pricing/src/models/linear.py
from sklearn.linear_model import LinearRegression

def train_ols(X, y):
    model = LinearRegression()
    model.fit(X, y)
    return model

Overwriting /content/drive/MyDrive/ml_asset_pricing/src/models/linear.py


In [ ]:
%%writefile /content/drive/MyDrive/ml_asset_pricing/src/models/elastic_net.py
from sklearn.linear_model import ElasticNet

def train_enet(X, y, alpha=0.0001, l1_ratio=0.3):
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
    model.fit(X, y)
    return model

Overwriting /content/drive/MyDrive/ml_asset_pricing/src/models/elastic_net.py


In [6]:
%%writefile /content/drive/MyDrive/ml_asset_pricing/src/models/tree.py
from sklearn.ensemble import RandomForestRegressor

def train_rf(X, y, n_estimators=200, max_depth=6, random_state=42):
    model = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,          # shallow
    min_samples_leaf=5,   # 작게
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

    model.fit(X, y)
    return model

Overwriting /content/drive/MyDrive/ml_asset_pricing/src/models/tree.py


In [ ]:
%%writefile /content/drive/MyDrive/ml_asset_pricing/src/evaluate.py
import numpy as np
import pandas as pd

# Out-of-sample R^2 (모델이 평균 예측보다 얼마나 잘 맞추는지)
# (금융에서는 R^2 주로 낮게 나옴 => 다른 지표 같이 봐야함)
def oos_r2(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = ((y_true - y_true.mean()) ** 2).mean()
    if denom == 0:
        return np.nan
    return 1 - ((y_true - y_pred) ** 2).mean() / denom

# 상위 10% = long / 하위 10% = short 포트폴리오 수익률
def long_short_portfolio(df, pred_col="pred", target_col="target", long_q=0.9, short_q=0.1):
    df = df.copy()
    df = df.dropna(subset=[pred_col, target_col])

    df["rank"] = df.groupby("date")[pred_col].rank(pct=True)

    long = df[df["rank"] >= long_q]
    short = df[df["rank"] <= short_q]

    long_ret = long.groupby("date")[target_col].mean()
    short_ret = short.groupby("date")[target_col].mean()

    ls = (long_ret - short_ret).dropna()
    ls.name = "long_short_return"
    return ls

# Sharpe Ratio (위험 대비 수익)( (수익률-무위험수익률)/변동성 )
def sharpe(returns):
    returns = pd.Series(returns).dropna()
    if returns.std() == 0:
        return np.nan
    return returns.mean() / returns.std() * (252 ** 0.5)

Overwriting /content/drive/MyDrive/ml_asset_pricing/src/evaluate.py


In [ ]:
%%writefile /content/drive/MyDrive/ml_asset_pricing/src/train.py
import pandas as pd
from src.data_loader import load_data
from src.data_loader import load_data_fixed
from src.data_loader import sample_universe
from src.feature_engineering import add_features
from src.dataset import create_target, normalize_features, split_data, get_feature_cols
from src.models.linear import train_ols
from src.models.elastic_net import train_enet
from src.models.tree import train_rf
from src.evaluate import oos_r2, long_short_portfolio, sharpe

def prepare_data(mode="baseline"):
    if mode == "baseline":
      tickers = [
      'AAPL','MSFT','GOOG','AMZN','META','NVDA','TSLA','JPM','V','PG',
      'ADBE','CRM','ORCL','INTC','AMD',
      'JNJ','PFE','MRK','ABBV','TMO',
      'KO','PEP','COST','WMT','HD',
      'XOM','CVX','BA','CAT','GE',
      'NFLX','DIS','MA','PYPL','BRK-B','GS','UPS','UNP','LIN','AMAT']
      df = load_data_fixed(tickers)
    else:
      df = load_data()
    print("1 clear")

    df = add_features(df,mode)
    print("2 clear")

    df = create_target(df)
    print("3 clear")

    if mode == "improved":
      df = sample_universe(df, n=100, method="random")
      print("sampling clear")

    df = df.dropna(subset=["target"]).reset_index(drop=True)

    feature_cols = get_feature_cols(df)

    df = normalize_features(df, feature_cols)

    train, val, test = split_data(df)

    train = train.dropna(subset=feature_cols + ["target"]).copy()
    val = val.dropna(subset=feature_cols + ["target"]).copy()
    test = test.dropna(subset=feature_cols + ["target"]).copy()

    return train, val, test, feature_cols

def run_model(train, val, test, feature_cols, model_name="rf"):

    X_train = train[feature_cols].values
    y_train = train["target"].values

    X_test = test[feature_cols].values
    y_test = test["target"].values

    if model_name == "ols":
        model = train_ols(X_train, y_train)
    elif model_name == "enet":
        model = train_enet(X_train, y_train)
    elif model_name == "rf":
        model = train_rf(X_train, y_train)
    else:
        raise ValueError(f"Unknown model_name: {model_name}")

    test = test.copy()
    test["pred"] = model.predict(X_test)

    r2 = oos_r2(y_test, test["pred"].values)
    ls = long_short_portfolio(test, pred_col="pred", target_col="target")
    sh = sharpe(ls)

    return {
        "model": model_name,
        "oos_r2": r2,
        "sharpe": sh,
    }

Overwriting /content/drive/MyDrive/ml_asset_pricing/src/train.py


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/ml_asset_pricing")

from src.train import prepare_data, run_model



train1, val1, test1, feature_cols1 = prepare_data(mode="baseline")
train2, val2, test2, feature_cols2 = prepare_data(mode="improved")

print("baseline run")
results1_ols = run_model(train1, val1, test1, feature_cols1, "ols")
results1_enet = run_model(train1, val1, test1, feature_cols1, "enet")
results1_rf = run_model(train1, val1, test1, feature_cols1, "rf")

print()
print("improved run")
results2_ols = run_model(train2, val2, test2, feature_cols2, "ols")
results2_enet = run_model(train2, val2, test2, feature_cols2, "enet")
results2_rf = run_model(train2, val2, test2, feature_cols2, "rf")

1 clear
2 clear
3 clear


ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['SNDK', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2010-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1262322000, endDate = 1735621200")')


1 clear
2 clear
3 clear
sampling clear
baseline run

improved run


In [ ]:
print("baseline result")
print(results1_ols)
print(results1_enet)
print(results1_rf)

print()

print("improved result")
print(results2_ols)
print(results2_enet)
print(results2_rf)

baseline result
{'model': 'ols', 'oos_r2': np.float64(5.058360520404648e-05), 'sharpe': np.float64(0.6504379209902983)}
{'model': 'enet', 'oos_r2': np.float64(0.00017148339274930535), 'sharpe': np.float64(1.1806136898244644)}
{'model': 'rf', 'oos_r2': np.float64(-0.0003203911758953293), 'sharpe': np.float64(0.6550431013230821)}

improved result
{'model': 'ols', 'oos_r2': np.float64(3.0294786150908415e-05), 'sharpe': np.float64(1.0685728845466904)}
{'model': 'enet', 'oos_r2': np.float64(5.3095466334251995e-05), 'sharpe': np.float64(1.0940437157113423)}
{'model': 'rf', 'oos_r2': np.float64(-8.659256138465743e-05), 'sharpe': np.float64(0.5928679040578153)}
